## 实验8.2 CANN 云平台机器人开发实验案例

> 结合第8章智能机器人系统开发理论与香橙派实验，在 CANN 云开发平台上运行演示

机器人作为实体物件，不太好在云平台上直接操作。本 notebook 将第8章的理论知识与香橙派实验的核心概念**数字化**，在 CANN 云平台上用代码仿真完整的'感知-建图-规划-控制-学习'链路，让你在云端就能深入理解智能机器人系统的每一个环节，以及**昇腾 NPU 在机器人智能中的加速作用**。

**运行环境**：`cann_9.0.0-py3.11-A2-arm-20260715` · `ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB`

---

## 目录

1. 环境检查：昇腾 NPU 与 CANN
2. 机器人运动学：差速底盘仿真
3. 传感器仿真：里程计与激光雷达
4. SLAM 建图：占据栅格地图构建
5. 自主定位：AMCL 粒子滤波
6. 路径规划：A* 与 Nav2 仿真
7. 深度强化学习：DQN 避障训练（NPU 加速）
8. 端到端演示：完整导航闭环
9. 昇腾 NPU 加速对比
10. 课后练习

---

## 1. 环境检查：昇腾 NPU 与 CANN

首先检查当前云平台的昇腾 NPU 环境是否就绪。

In [ ]:
import sys, os, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print(f'Python 版本: {sys.version}')
print(f'NumPy 版本: {np.__version__}')
print(f'Matplotlib 版本: {matplotlib.__version__}')
print(f'当前工作目录: {os.getcwd()}')
print(f'CPU 核心数: {os.cpu_count()}')
print()

# 检查 PyTorch
try:
    import torch
    print(f'PyTorch 版本: {torch.__version__}')
except ImportError:
    print('PyTorch 未安装')

# 检查昇腾 NPU
npu_available = False
try:
    import torch_npu
    npu_available = torch.npu.is_available()
    print(f'torch_npu 已导入')
    print(f'NPU 是否可用: {npu_available}')
    if npu_available:
        print(f'NPU 设备数量: {torch.npu.device_count()}')
        print(f'NPU 设备名: {torch.npu.get_device_name(0)}')
except ImportError:
    print('torch_npu 未安装（非昇腾环境）')

device = torch.device('npu:0') if npu_available else torch.device('cpu')
print(f'\n计算设备: {device}')
print(f'NPU 加速: {"已启用" if npu_available else "未启用（使用CPU）"}')

In [ ]:
# 检查 CANN 环境变量
print('=== CANN 环境变量 ===')
cann_vars = ['ASCEND_HOME_PATH', 'LD_LIBRARY_PATH', 'PYTHONPATH', 'TOOLKIT_HOME', 'ATC_HOME']
for var in cann_vars:
    val = os.environ.get(var, '')
    if val:
        print(f'  {var}: {val[:80]}...' if len(val) > 80 else f'  {var}: {val}')
    else:
        print(f'  {var}: (未设置)')

# 检查 ATC 工具
print()
print('=== ATC 模型转换工具 ===')
import shutil
atc_path = shutil.which('atc')
if atc_path:
    print(f'  ATC 路径: {atc_path}')
else:
    print('  ATC 未在 PATH 中（可能需要 source 环境变量）')

---

## 2. 机器人运动学：差速底盘仿真

对应第8章 8.2 节与香橙派实验的运动控制链路。我们仿真差速底盘的完整运动学模型，模拟真机上 `/cmd_vel -> 运动学解算 -> 电机 -> /odom` 的过程。

In [ ]:
class DifferentialDriveRobot:
    '''差速底盘完整仿真（对应真机 crobot_control）'''
    def __init__(self, wheel_base=0.28, wheel_radius=0.06):
        self.L = wheel_base      # 轮距
        self.r = wheel_radius    # 轮半径
        self.x = 0.0; self.y = 0.0; self.theta = 0.0
        self.v = 0.0; self.omega = 0.0
        self.v_left = 0.0; self.v_right = 0.0
        self.odom_history = []
    
    def cmd_vel(self, v, omega):
        '''接收 /cmd_vel 指令，运动学逆解算'''
        self.v = v; self.omega = omega
        self.v_left = v - omega * self.L / 2
        self.v_right = v + omega * self.L / 2
    
    def update(self, dt=0.05):
        '''更新位姿，生成 /odom'''
        self.x += self.v * np.cos(self.theta) * dt
        self.y += self.v * np.sin(self.theta) * dt
        self.theta += self.omega * dt
        self.odom_history.append({
            'x': self.x, 'y': self.y, 'theta': self.theta,
            'v': self.v, 'omega': self.omega,
            'v_left': self.v_left, 'v_right': self.v_right
        })
    
    def get_odom(self):
        '''返回当前 /odom 消息内容'''
        return self.x, self.y, self.theta

# 创建机器人
robot = DifferentialDriveRobot()
print('差速底盘仿真器已创建')
print(f'  轮距 L = {robot.L} m')
print(f'  轮半径 r = {robot.r} m')
print()

# 模拟一段运动：前进 -> 左转 -> 前进 -> 右转 -> 停止
motion_sequence = [
    (0.2, 0.0, 60),     # 前进 3秒
    (0.15, 0.3, 40),    # 左转前进 2秒
    (0.2, 0.0, 60),     # 前进 3秒
    (0.15, -0.3, 40),   # 右转前进 2秒
    (0.1, 0.5, 30),     # 原地左转 1.5秒
    (0.0, 0.0, 10),     # 停止
]

print('=== 运动控制序列 ===')
for v, w, n in motion_sequence:
    robot.cmd_vel(v, w)
    for _ in range(n):
        robot.update()
    x, y, th = robot.get_odom()
    print(f'  cmd_vel(v={v:.2f}, w={w:.1f}) -> odom(x={x:.3f}, y={y:.3f}, theta={np.degrees(th):.1f}deg)')
    print(f'    左轮={robot.v_left:.4f} m/s, 右轮={robot.v_right:.4f} m/s')

In [ ]:
# 可视化里程计轨迹
odom = np.array([[h['x'], h['y']] for h in robot.odom_history])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 轨迹图
axes[0].plot(odom[:, 0], odom[:, 1], 'b-', linewidth=2)
axes[0].plot(odom[0, 0], odom[0, 1], 'go', markersize=12, label='Start')
axes[0].plot(odom[-1, 0], odom[-1, 1], 'r^', markersize=12, label='End')
# 标注方向变化点
cumsum = 0
for v, w, n in motion_sequence:
    cumsum += n
    if cumsum < len(odom):
        axes[0].plot(odom[cumsum, 0], odom[cumsum, 1], 'kx', markersize=8)
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title('/odom Pose Trajectory'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# 速度曲线
times = np.arange(len(robot.odom_history)) * 0.05
velocities = np.array([[h['v'], h['omega'], h['v_left'], h['v_right']] for h in robot.odom_history])
axes[1].plot(times, velocities[:, 0], label='Linear vel v (m/s)')
axes[1].plot(times, velocities[:, 1], label='Angular vel omega (rad/s)')
axes[1].plot(times, velocities[:, 2], label='Left wheel v_left', alpha=0.5)
axes[1].plot(times, velocities[:, 3], label='Right wheel v_right', alpha=0.5)
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Velocity')
axes[1].set_title('/cmd_vel Velocity Command'); axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Differential Drive Sim: /cmd_vel -> /odom Full Chain', fontsize=14)
plt.tight_layout()
os.makedirs('./images', exist_ok=True)
plt.savefig('./images/cloud_diff_drive.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 3. 传感器仿真：里程计与激光雷达

对应香橙派实验中的传感器驱动。我们仿真激光雷达的 ToF 测距和里程计的累积误差。

In [ ]:
class LaserScanner:
    '''仿真单线激光雷达（对应真机 LSLIDAR N10）'''
    def __init__(self, range_max=10.0, n_rays=180, noise_std=0.02):
        self.range_max = range_max
        self.n_rays = n_rays
        self.noise_std = noise_std
        self.angles = np.linspace(-np.pi, np.pi, n_rays)
    
    def scan(self, robot_pos, robot_theta, obstacles):
        '''模拟激光扫描，返回 /scan 消息内容'''
        ranges = np.full(self.n_rays, self.range_max)
        for i, angle in enumerate(self.angles):
            world_angle = angle + robot_theta
            dx = np.cos(world_angle); dy = np.sin(world_angle)
            for obs in obstacles:
                # 射线与圆形障碍物求交
                ox, oy, r = obs
                t = (ox - robot_pos[0]) * dx + (oy - robot_pos[1]) * dy
                if t > 0:
                    closest_x = robot_pos[0] + t * dx
                    closest_y = robot_pos[1] + t * dy
                    dist_to_center = np.sqrt((closest_x-ox)**2 + (closest_y-oy)**2)
                    if dist_to_center < r:
                        hit_dist = t - np.sqrt(r**2 - dist_to_center**2)
                        if 0 < hit_dist < ranges[i]:
                            ranges[i] = hit_dist
            ranges[i] += np.random.randn() * self.noise_std
        return ranges

# 创建激光雷达
laser = LaserScanner(range_max=10.0, n_rays=180)

# 定义环境中的障碍物 (x, y, radius)
obstacles = [
    (3, 2, 0.5), (5, 5, 0.8), (7, 3, 0.6),
    (2, 7, 0.5), (6, 8, 0.7), (8, 6, 0.5),
    (4, 4, 0.3), (1, 5, 0.4)
]

# 在 (0, 0) 位置扫描
ranges = laser.scan([0, 0], 0, obstacles)
print(f'激光雷达: {laser.n_rays} 线, 量程 {laser.range_max}m')
print(f'扫描结果: {len(ranges)} 个距离值')
print(f'最近障碍: {np.min(ranges):.3f} m')
print(f'最远距离: {np.max(ranges):.3f} m')
print(f'有效回波: {np.sum(ranges < laser.range_max)} / {laser.n_rays}')

In [ ]:
# 可视化激光扫描结果
fig, ax = plt.subplots(figsize=(8, 8))

# 绘制障碍物
for ox, oy, r in obstacles:
    circle = plt.Circle((ox, oy), r, color='gray', alpha=0.5)
    ax.add_patch(circle)

# 绘制激光射线
valid = ranges < laser.range_max
for i in range(laser.n_rays):
    if valid[i]:
        angle = laser.angles[i]
        ax.plot([0, ranges[i]*np.cos(angle)], [0, ranges[i]*np.sin(angle)], 
                'r-', alpha=0.1, linewidth=0.5)

# 绘制激光点云
points_x = ranges[valid] * np.cos(laser.angles[valid])
points_y = ranges[valid] * np.sin(laser.angles[valid])
ax.scatter(points_x, points_y, c='red', s=5, label='LiDAR points /scan')

ax.plot(0, 0, 'b^', markersize=12, label='Robot')
ax.set_xlim(-1, 11); ax.set_ylim(-1, 11)
ax.set_aspect('equal')
ax.set_title('LiDAR Scan Sim (/scan topic)', fontsize=14)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('./images', exist_ok=True)
plt.savefig('./images/cloud_lidar_scan.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 4. SLAM 建图：占据栅格地图构建

对应第8章 8.3 节与香橙派实验的 slam_toolbox 建图。我们仿真机器人边运动边建图的过程。

In [ ]:
class SimpleSLAM:
    '''简化的 2D 栅格 SLAM 仿真'''
    def __init__(self, grid_size=30, resolution=0.5):
        self.grid_size = grid_size
        self.resolution = resolution  # m/cell
        # 占据概率: 0=空闲, 0.5=未知, 1=占据
        self.grid = np.ones((grid_size, grid_size)) * 0.5
        self.laser = LaserScanner(range_max=6.0, n_rays=120, noise_std=0.05)
    
    def update(self, robot_pos, robot_theta, obstacles):
        '''一帧激光扫描更新地图'''
        rx, ry = robot_pos
        ranges = self.laser.scan(robot_pos, robot_theta, obstacles)
        
        for i, angle in enumerate(self.laser.angles):
            world_angle = angle + robot_theta
            dx = np.cos(world_angle); dy = np.sin(world_angle)
            dist = ranges[i]
            
            # 沿射线标记空闲区域
            n_steps = int(dist / self.resolution)
            for s in range(n_steps):
                px = rx + s * self.resolution * dx
                py = ry + s * self.resolution * dy
                gx = int(px / self.resolution + self.grid_size / 2)
                gy = int(py / self.resolution + self.grid_size / 2)
                if 0 <= gx < self.grid_size and 0 <= gy < self.grid_size:
                    self.grid[gx, gy] = max(0, self.grid[gx, gy] - 0.1)
            
            # 标记端点为占据
            if dist < self.laser.range_max:
                px = rx + dist * dx; py = ry + dist * dy
                gx = int(px / self.resolution + self.grid_size / 2)
                gy = int(py / self.resolution + self.grid_size / 2)
                if 0 <= gx < self.grid_size and 0 <= gy < self.grid_size:
                    self.grid[gx, gy] = min(1, self.grid[gx, gy] + 0.3)

slam = SimpleSLAM(grid_size=30, resolution=0.5)

# 定义环境
room_obstacles = [
    # 墙壁（用小圆形排列模拟）
    *[(x, 0, 0.3) for x in np.arange(0, 12, 0.6)],       # 下墙
    *[(x, 10, 0.3) for x in np.arange(0, 12, 0.6)],      # 上墙
    *[(0, y, 0.3) for y in np.arange(0, 10, 0.6)],       # 左墙
    *[(12, y, 0.3) for y in np.arange(0, 10, 0.6)],      # 右墙
    # 内部障碍
    (4, 4, 0.5), (8, 6, 0.6), (3, 7, 0.4), (9, 3, 0.5),
]

# 机器人沿路径运动并建图
waypoints = [(1, 1), (1, 8), (5, 8), (5, 2), (10, 2), (10, 8), (6, 5), (1, 1)]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
show_steps = [1, 4, 7]

for wp_idx in range(len(waypoints) - 1):
    start_wp = np.array(waypoints[wp_idx])
    end_wp = np.array(waypoints[wp_idx + 1])
    direction = end_wp - start_wp
    distance = np.linalg.norm(direction)
    n_steps = int(distance / 0.3)
    
    for s in range(n_steps):
        pos = start_wp + direction * (s / max(n_steps, 1))
        theta = np.arctan2(direction[1], direction[0])
        slam.update(pos, theta, room_obstacles)
    
    if wp_idx in show_steps:
        ax = axes[show_steps.index(wp_idx)]
        ax.imshow(slam.grid, cmap='gray_r', origin='lower', 
                  extent=[-slam.grid_size*slam.resolution/2, slam.grid_size*slam.resolution/2,
                          -slam.grid_size*slam.resolution/2, slam.grid_size*slam.resolution/2])
        ax.plot(pos[0], pos[1], 'b^', markersize=10)
        ax.set_title(f'Mapping Waypoint {wp_idx+1}', fontsize=13)

plt.suptitle('SLAM Mapping Sim: Map Built Incrementally with Robot Motion', fontsize=14)
plt.tight_layout()
os.makedirs('./images', exist_ok=True)
plt.savefig('./images/cloud_slam.png', dpi=150, bbox_inches='tight')
plt.show()
print('SLAM 建图完成！黑色=占据, 白色=空闲, 灰色=未知')

---

## 5. 自主定位：AMCL 粒子滤波

对应香橙派实验的 AMCL 定位。仿真粒子滤波的收敛过程。

In [ ]:
class AMCLSim:
    '''AMCL 粒子滤波定位仿真'''
    def __init__(self, n_particles=500, init_pos=(5, 5), init_std=2.0):
        self.n = n_particles
        self.particles = np.column_stack([
            np.random.normal(init_pos[0], init_std, n_particles),
            np.random.normal(init_pos[1], init_std, n_particles),
            np.random.uniform(-np.pi, np.pi, n_particles)
        ])
        self.weights = np.ones(n_particles) / n_particles
    
    def update(self, true_pos, true_theta, sensor_noise=0.3):
        '''用激光观测更新粒子权重'''
        for i in range(self.n):
            dist_err = np.sqrt((self.particles[i, 0]-true_pos[0])**2 + 
                              (self.particles[i, 1]-true_pos[1])**2)
            self.weights[i] = np.exp(-dist_err**2 / (2 * sensor_noise**2))
        self.weights /= self.weights.sum() + 1e-10
    
    def resample(self, motion_noise=0.15):
        '''重采样 + 运动模型噪声'''
        indices = np.random.choice(self.n, self.n, p=self.weights)
        self.particles = self.particles[indices]
        self.particles[:, 0] += np.random.randn(self.n) * motion_noise
        self.particles[:, 1] += np.random.randn(self.n) * motion_noise
        self.particles[:, 2] += np.random.randn(self.n) * 0.05
        self.weights = np.ones(self.n) / self.n
    
    def estimate(self):
        return np.average(self.particles, weights=self.weights, axis=0)

# 仿真 AMCL 定位
np.random.seed(42)
true_pos = np.array([7.0, 6.0]); true_theta = 0.5
amcl = AMCLSim(n_particles=500, init_pos=(5, 5), init_std=2.0)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
show_steps = [0, 5, 20]

for step in range(25):
    amcl.update(true_pos, true_theta)
    if step in show_steps:
        ax = axes[show_steps.index(step)]
        ax.scatter(amcl.particles[:, 0], amcl.particles[:, 1], c='red', s=5, alpha=0.3, label='Particles')
        ax.plot(true_pos[0], true_pos[1], 'b^', markersize=15, label='True pose')
        est = amcl.estimate()
        ax.plot(est[0], est[1], 'g*', markersize=15, label='Estimated pose')
        ax.set_title(f'AMCL Step {step}', fontsize=13)
        ax.legend(fontsize=9); ax.set_xlim(2, 10); ax.set_ylim(2, 10)
    amcl.resample()

est = amcl.estimate()
print(f'真实位置: ({true_pos[0]}, {true_pos[1]})')
print(f'估计位置: ({est[0]:.2f}, {est[1]:.2f})')
print(f'定位误差: {np.linalg.norm(est[:2] - true_pos):.3f} m')
plt.suptitle('AMCL Particle Filter Localization Sim', fontsize=14)
plt.tight_layout()
os.makedirs('./images', exist_ok=True)
plt.savefig('./images/cloud_amcl.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. 路径规划：A* 与 Nav2 仿真

对应第8章的导航方法与香橙派实验的 Nav2 导航。仿真全局规划（A*）和局部控制（DWA）。

In [ ]:
from heapq import heappush, heappop

class Nav2Sim:
    '''Nav2 导航仿真：全局规划(A*) + 局部控制(DWA)'''
    def __init__(self, grid):
        self.grid = grid
        self.rows, self.cols = grid.shape
    
    def global_plan(self, start, goal):
        '''A* 全局路径规划（对应 NavfnPlanner）'''
        open_set = [(0, start)]; came_from = {}; g_score = {start: 0}
        while open_set:
            _, current = heappop(open_set)
            if current == goal:
                path = [current]
                while current in came_from:
                    current = came_from[current]; path.append(current)
                return path[::-1]
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                nx, ny = current[0]+dx, current[1]+dy
                if 0<=nx<self.rows and 0<=ny<self.cols and self.grid[nx,ny]==0:
                    g = g_score[current] + np.sqrt(dx**2+dy**2)
                    if (nx,ny) not in g_score or g < g_score[(nx,ny)]:
                        came_from[(nx,ny)] = current; g_score[(nx,ny)] = g
                        f = g + np.sqrt((nx-goal[0])**2+(ny-goal[1])**2)
                        heappush(open_set, (f, (nx,ny)))
        return []
    
    def local_control(self, pos, path_idx, global_path, obstacles_dynamic=None):
        '''DWA 局部控制（简化版）'''
        if path_idx >= len(global_path):
            return pos, path_idx
        target = np.array(global_path[min(path_idx + 2, len(global_path)-1)])
        direction = target - np.array(pos)
        dist = np.linalg.norm(direction)
        if dist < 0.5:
            path_idx += 1
        if dist > 0:
            new_pos = np.array(pos) + direction / dist * 0.3
        else:
            new_pos = np.array(pos)
        return new_pos, path_idx

# 创建导航环境
nav_grid = np.zeros((20, 20))
nav_grid[0, :] = 1; nav_grid[-1, :] = 1; nav_grid[:, 0] = 1; nav_grid[:, -1] = 1
nav_grid[4:7, 3:12] = 1
nav_grid[10:14, 8:11] = 1
nav_grid[3:6, 14:17] = 1

nav = Nav2Sim(nav_grid)
start, goal = (1, 1), (18, 18)
global_path = nav.global_plan(start, goal)
print(f'全局路径规划(A*): {len(global_path)} 步')

# 仿真局部控制跟踪
robot_pos = np.array(start, dtype=float)
path_idx = 0
local_trace = [tuple(robot_pos)]

for _ in range(200):
    robot_pos, path_idx = nav.local_control(robot_pos, path_idx, global_path)
    local_trace.append(tuple(robot_pos))
    if np.linalg.norm(robot_pos - np.array(goal)) < 1.0:
        break

local_trace = np.array(local_trace)
print(f'局部控制跟踪: {len(local_trace)} 步')
print(f'最终位置: ({local_trace[-1, 0]:.1f}, {local_trace[-1, 1]:.1f}), 目标: {goal}')

In [ ]:
# 可视化导航过程
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(nav_grid, cmap='binary', origin='lower')

# 全局路径
if global_path:
    py, px = zip(*global_path)
    ax.plot(px, py, 'g-', linewidth=3, alpha=0.5, label='Global path (A*)')

# 局部轨迹
ax.plot(local_trace[:, 1], local_trace[:, 0], 'b-', linewidth=2, label='Local trajectory (DWA)')

ax.plot(start[1], start[0], 'go', markersize=15, label='Start')
ax.plot(goal[1], goal[0], 'r^', markersize=15, label='Goal')

ax.set_title('Nav2 Navigation Sim: Global Plan (A*) + Local Control (DWA)', fontsize=14)
ax.legend(fontsize=11, loc='upper left')
plt.tight_layout()
os.makedirs('./images', exist_ok=True)
plt.savefig('./images/cloud_nav2.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 7. 深度强化学习：DQN 避障训练（NPU 加速）

对应第8章 8.4 节。在昇腾 NPU 上训练 Dueling DQN 避障智能体，体验 NPU 对强化学习训练的加速。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# 确定设备
device = torch.device('cpu')
try:
    import torch_npu
    if torch.npu.is_available():
        device = torch.device('npu:0')
except ImportError:
    pass

# Dueling DQN 网络
class DuelingDQN(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=128):
        super().__init__()
        self.feature = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU(),
                                      nn.Linear(hidden, hidden), nn.ReLU())
        self.value = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, 1))
        self.advantage = nn.Sequential(nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, n_actions))
    def forward(self, x):
        f = self.feature(x)
        v = self.value(f)
        a = self.advantage(f)
        return v + a - a.mean(dim=1, keepdim=True)

# 简化避障环境
class ObstacleEnv:
    def __init__(self, size=12):
        self.size = size; self.n_actions = 9
    def reset(self):
        self.pos = np.array([0, 0]); self.goal = np.array([self.size-1, self.size-1])
        self.obstacles = set()
        for _ in range(20):
            self.obstacles.add(tuple(np.random.randint(1, self.size-1, 2)))
        return self._state()
    def _state(self):
        d = []
        for dx, dy in [(0,1),(-1,0),(1,0),(0,-1)]:
            dist = 0
            for s in range(1, self.size):
                nx, ny = self.pos[0]+dx*s, self.pos[1]+dy*s
                if nx<0 or nx>=self.size or ny<0 or ny>=self.size or (nx,ny) in self.obstacles:
                    dist = s; break
            else: dist = self.size
            d.append(dist / self.size)
        g = (self.goal - self.pos) / self.size
        return np.array(d + [self.pos[0]/self.size, self.pos[1]/self.size, g[0], g[1]], dtype=np.float32)
    def step(self, action):
        moves = {0:(0,0), 1:(-1,0), 2:(1,0), 3:(0,1), 4:(0,-1),
                 5:(-1,1), 6:(1,1), 7:(-1,-1), 8:(1,-1)}
        dx, dy = moves[action]
        new_pos = self.pos + np.array([dx, dy])
        collision = (new_pos[0]<0 or new_pos[0]>=self.size or new_pos[1]<0 or new_pos[1]>=self.size
                     or tuple(new_pos) in self.obstacles)
        if not collision: self.pos = new_pos
        reward = -6.0 if collision else (10.0 if np.array_equal(self.pos, self.goal) else (-0.1 if action==0 else 0.2))
        done = collision or np.array_equal(self.pos, self.goal)
        return self._state(), reward, done

env = ObstacleEnv(12)
state_dim = 8; n_actions = 9
policy_net = DuelingDQN(state_dim, n_actions).to(device)
target_net = DuelingDQN(state_dim, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
buffer = deque(maxlen=3000)

print(f'训练设备: {device}')
print(f'网络参数: {sum(p.numel() for p in policy_net.parameters()):,}')
print(f'环境: {env.size}x{env.size} 栅格, {n_actions} 个动作')
print()

# 训练
N_EPISODES = 150; BATCH = 64; GAMMA = 0.99
rewards_history = []
start_time = time.time()

for ep in range(N_EPISODES):
    state = env.reset(); total_r = 0
    for t in range(80):
        eps = 0.05 + 0.95 * np.exp(-ep / 50)
        if random.random() < eps:
            action = random.randint(0, n_actions-1)
        else:
            with torch.no_grad():
                action = policy_net(torch.FloatTensor(state).unsqueeze(0).to(device)).argmax().item()
        next_state, reward, done = env.step(action)
        buffer.append((state, action, reward, next_state, done))
        state = next_state; total_r += reward
        if len(buffer) >= BATCH:
            batch = random.sample(buffer, BATCH)
            s, a, r, s2, d = zip(*batch)
            s_t = torch.FloatTensor(np.array(s)).to(device)
            a_t = torch.LongTensor(a).to(device)
            r_t = torch.FloatTensor(r).to(device)
            s2_t = torch.FloatTensor(np.array(s2)).to(device)
            d_t = torch.FloatTensor(d).to(device)
            q = policy_net(s_t).gather(1, a_t.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                tq = r_t + GAMMA * target_net(s2_t).max(1)[0] * (1-d_t)
            loss = nn.functional.mse_loss(q, tq)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        if done: break
    if ep % 10 == 0: target_net.load_state_dict(policy_net.state_dict())
    rewards_history.append(total_r)
    if (ep+1) % 50 == 0:
        print(f'回合 {ep+1}/{N_EPISODES}, 平均奖励: {np.mean(rewards_history[-50:]):.2f}, eps: {eps:.3f}')

elapsed = time.time() - start_time
print(f'\n训练完成! 耗时: {elapsed:.1f}s (设备: {device})')

In [ ]:
# 绘制训练曲线
plt.figure(figsize=(10, 5))
plt.plot(rewards_history, alpha=0.3, color='blue')
window = 20
if len(rewards_history) >= window:
    smoothed = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(rewards_history)), smoothed, color='red', linewidth=2, label='Moving average')
plt.xlabel('Training Episode'); plt.ylabel('Cumulative Reward')
plt.title(f'Dueling DQN Obstacle Avoidance Training (Device: {device})', fontsize=14)
plt.legend(); plt.axhline(y=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('./images', exist_ok=True)
plt.savefig('./images/cloud_dqn_train.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 8. 端到端演示：完整导航闭环

将以上所有模块串联，演示完整的'感知-建图-定位-规划-控制'闭环。

In [ ]:
print('=' * 60)
print('CANN 云平台机器人开发：端到端演示')
print('=' * 60)
print()

# Step 1: 环境检查
print('[Step 1] 环境检查')
print(f'  设备: {device}')
print(f'  NPU 加速: {"已启用" if npu_available else "未启用"}')
print()

# Step 2: 运动学仿真
print('[Step 2] 差速底盘运动学仿真')
robot2 = DifferentialDriveRobot()
robot2.cmd_vel(0.2, 0.0)
for _ in range(50): robot2.update()
x, y, th = robot2.get_odom()
print(f'  /cmd_vel(0.2, 0.0) -> /odom(x={x:.3f}, y={y:.3f})')
print()

# Step 3: SLAM 建图
print('[Step 3] SLAM 建图仿真')
slam2 = SimpleSLAM(grid_size=20, resolution=0.5)
for angle in np.linspace(0, 2*np.pi, 50):
    pos = [3*np.cos(angle), 3*np.sin(angle)]
    slam2.update(pos, angle + np.pi/2, room_obstacles[:10])
print(f'  建图完成: {slam2.grid_size}x{slam2.grid_size} 栅格地图')
print()

# Step 4: AMCL 定位
print('[Step 4] AMCL 定位仿真')
amcl2 = AMCLSim(n_particles=200, init_pos=(3, 3), init_std=1.5)
true_p = np.array([4.0, 4.0])
for _ in range(15):
    amcl2.update(true_p, 0); amcl2.resample()
est = amcl2.estimate()
print(f'  真实: ({true_p[0]}, {true_p[1]}), 估计: ({est[0]:.2f}, {est[1]:.2f})')
print()

# Step 5: 路径规划
print('[Step 5] A* 路径规划')
nav2 = Nav2Sim(nav_grid)
path2 = nav2.global_plan((1, 1), (18, 18))
print(f'  路径长度: {len(path2)} 步')
print()

# Step 6: DQN 避障
print('[Step 6] DQN 避障推理')
test_state = env.reset()
with torch.no_grad():
    q_vals = policy_net(torch.FloatTensor(test_state).unsqueeze(0).to(device))
    best_action = q_vals.argmax().item()
print(f'  Q值: {q_vals.cpu().numpy().flatten()}')
print(f'  最优动作: {best_action}')
print()

print('=' * 60)
print('端到端演示完成！')
print('  感知(激光) -> 建图(SLAM) -> 定位(AMCL) -> 规划(A*) -> 控制(/cmd_vel) -> 避障(DQN)')
print('=' * 60)

---

## 9. 昇腾 NPU 加速对比

对比 CPU 与 NPU 在机器人相关计算上的性能差异。

In [ ]:
import time

print('=== CPU vs NPU 性能对比 ===')
print()

# 测试1: 大规模矩阵运算（模拟批量运动学解算）
N = 5_000_000
v_batch = np.random.uniform(0, 0.5, N)
w_batch = np.random.uniform(-0.5, 0.5, N)
L = 0.28

start = time.time()
for _ in range(10):
    v_l = v_batch - w_batch * L / 2
    v_r = v_batch + w_batch * L / 2
cpu_t1 = (time.time() - start) / 10
print(f'[测试1] 批量运动学解算 ({N} 组):')
print(f'  CPU: {cpu_t1*1000:.2f} ms')

if npu_available:
    v_t = torch.from_numpy(v_batch).npu()
    w_t = torch.from_numpy(w_batch).npu()
    torch.npu.synchronize()
    start = time.time()
    for _ in range(10):
        v_l = v_t - w_t * L / 2
        v_r = v_t + w_t * L / 2
    torch.npu.synchronize()
    npu_t1 = (time.time() - start) / 10
    print(f'  NPU: {npu_t1*1000:.2f} ms')
    print(f'  加速比: {cpu_t1/npu_t1:.1f}x')
print()

# 测试2: 神经网络推理（模拟 DQN 避障决策）
batch_size = 10000
states = np.random.randn(batch_size, state_dim).astype(np.float32)

start = time.time()
for _ in range(100):
    with torch.no_grad():
        _ = policy_net.cpu()(torch.from_numpy(states))
cpu_t2 = (time.time() - start) / 100
print(f'[测试2] DQN 批量推理 (batch={batch_size}):')
print(f'  CPU: {cpu_t2*1000:.2f} ms')

if npu_available:
    policy_net.npu()
    states_t = torch.from_numpy(states).npu()
    torch.npu.synchronize()
    start = time.time()
    for _ in range(100):
        with torch.no_grad():
            _ = policy_net(states_t)
    torch.npu.synchronize()
    npu_t2 = (time.time() - start) / 100
    print(f'  NPU: {npu_t2*1000:.2f} ms')
    print(f'  加速比: {cpu_t2/npu_t2:.1f}x')
    policy_net.cpu()
print()

# 测试3: 矩阵乘法（模拟 SLAM 图优化）
M = 2000
A = np.random.randn(M, M).astype(np.float32)
B = np.random.randn(M, M).astype(np.float32)

start = time.time()
_ = A @ B
cpu_t3 = time.time() - start
print(f'[测试3] 矩阵乘法 ({M}x{M}):')
print(f'  CPU: {cpu_t3*1000:.2f} ms')

if npu_available:
    A_t = torch.from_numpy(A).npu()
    B_t = torch.from_numpy(B).npu()
    torch.npu.synchronize()
    start = time.time()
    _ = A_t @ B_t
    torch.npu.synchronize()
    npu_t3 = time.time() - start
    print(f'  NPU: {npu_t3*1000:.2f} ms')
    print(f'  加速比: {cpu_t3/npu_t3:.1f}x')

print()
print('结论: 对于大规模并行计算，NPU 相比 CPU 有显著加速优势。')
print('在真实机器人系统中，NPU 用于感知类AI任务（目标检测/语义分割），')
print('CPU 用于导航控制类任务（SLAM/Nav2），构成异构计算分工。')

---

## 10. 课后练习

请根据本演示案例完成以下题目进行自测。

**第1题**（单选题）本演示案例运行在什么云平台上？


- A. AWS
- B. cann_9.0.0-py3.11-A2-arm-20260715, ASCEND 1*NPU 910B3
- C. Azure
- D. Google Cloud


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）差速底盘运动学中，/cmd_vel到/odom的链路顺序是？


- A. /odom -> 运动学 -> /cmd_vel
- B. /cmd_vel -> 运动学解算 -> 电机 -> 编码器 -> /odom
- C. /cmd_vel -> /odom -> 电机
- D. 电机 -> /cmd_vel -> /odom


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）SLAM建图仿真中，占据栅格地图的三个状态是？


- A. 红、绿、蓝
- B. 占据(黑)、空闲(白)、未知(灰)
- C. 高、中、低
- D. 正、负、零


In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）AMCL粒子滤波定位的核心步骤是？


- A. 初始化->训练->推理
- B. 更新权重->重采样->运动模型预测
- C. 建图->保存->加载
- D. 规划->控制->反馈


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）Nav2导航栈的全局规划器使用的算法是？


- A. RRT
- B. A*/Dijkstra
- C. DWA
- D. TEB


In [ ]:
q5 = ''  # 填入你的选项，如 'B'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）Dueling DQN将Q值分解为？


- A. 策略和价值
- B. 价值流V(s)和优势流A(s,a)
- C. 编码和解码
- D. 前向和后向


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）本案例中NPU加速最显著的计算任务是？


- A. 小规模标量运算
- B. 大规模矩阵乘法和神经网络推理
- C. 字符串处理
- D. 文件IO


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）机器人系统中'导航在CPU、感知在NPU'的分工称为？


- A. 串行计算
- B. 异构计算
- C. 分布式计算
- D. 并行计算


In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）端到端演示的完整链路是？


- A. 感知->建图->定位->规划->控制->避障
- B. 训练->推理->部署
- C. 建图->保存->加载
- D. 规划->执行->反馈


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）在昇腾NPU上训练DQN时，需要调用什么函数等待异步计算完成？


- A. torch.cuda.synchronize()
- B. torch.npu.synchronize()
- C. time.sleep()
- D. torch.wait()


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**第11题**（单选题）激光雷达基于什么原理测距？


- A. 三角测距
- B. 飞行时间(ToF)
- C. 超声波
- D. 红外反射


In [ ]:
q11 = ''  # 填入你的选项，如 'B'
print(f'第11题答案已记录：{q11}' if q11 else '请填入答案并运行本单元格')

**第12题**（单选题）本案例将机器人实验'数字化'的核心思路是？


- A. 直接在云平台控制真机
- B. 用代码仿真完整感知-建图-规划-控制-学习链路
- C. 只做理论讲解不写代码
- D. 用VR模拟机器人


In [ ]:
q12 = ''  # 填入你的选项，如 'B'
print(f'第12题答案已记录：{q12}' if q12 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path
for candidate in (Path.cwd() / 'answer', Path.cwd() / '08_robot_dev' / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_04 import grade
grade(globals())

---

## 总结

本 notebook 将第8章智能机器人系统开发的理论知识与香橙派实验的核心概念**数字化**，在 CANN 云平台上用代码仿真了完整的机器人技术栈：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">对应章节/实验</th>
<th style="text-align: left;">云平台实现</th>
</tr>
<tr>
<td style="text-align: left;">差速底盘运动学</td>
<td style="text-align: left;">8.2 / 香橙派实验</td>
<td style="text-align: left;"><code>DifferentialDriveRobot</code> 仿真</td>
</tr>
<tr>
<td style="text-align: left;">激光雷达感知</td>
<td style="text-align: left;">8.3 / 香橙派实验</td>
<td style="text-align: left;"><code>LaserScanner</code> ToF 仿真</td>
</tr>
<tr>
<td style="text-align: left;">SLAM 建图</td>
<td style="text-align: left;">8.3 / 香橙派实验</td>
<td style="text-align: left;"><code>SimpleSLAM</code> 栅格地图构建</td>
</tr>
<tr>
<td style="text-align: left;">AMCL 定位</td>
<td style="text-align: left;">8.3 / 香橙派实验</td>
<td style="text-align: left;"><code>AMCLSim</code> 粒子滤波</td>
</tr>
<tr>
<td style="text-align: left;">路径规划</td>
<td style="text-align: left;">8.3 / 香橙派实验</td>
<td style="text-align: left;"><code>Nav2Sim</code> A* + DWA</td>
</tr>
<tr>
<td style="text-align: left;">DQN 避障</td>
<td style="text-align: left;">8.4</td>
<td style="text-align: left;"><code>DuelingDQN</code> NPU 训练</td>
</tr>
<tr>
<td style="text-align: left;">NPU 加速</td>
<td style="text-align: left;">昇腾平台</td>
<td style="text-align: left;">CPU vs NPU 对比</td>
</tr>
</table>

**核心收获**：
1. 理解了智能机器人'感知-建图-定位-规划-控制-学习'的完整闭环
2. 体验了昇腾 NPU 在大规模并行计算上的加速优势
3. 掌握了'导航在 CPU、感知在 NPU'的异构计算分工思想
4. 了解了从理论(PPT)到真机(香橙派)到云端(CANN)的完整学习路径

## 参考资料

- [第8章 PPT](./第8章-智能机器人系统开发v2.pptx)
- [香橙派实验手册](./exp8_ascend_orangepi_local/)
- [ROS2 官方文档](https://docs.ros.org/en/rolling/)
- [Nav2 导航框架](https://navigation.ros.org/)
- [SLAM Toolbox](https://github.com/SteveMacenski/slam_toolbox)
- [昇腾 CANN 文档](https://hiascend.com/document)

> 上一节：[03_ascend_orangepi_experiment.ipynb](./03_ascend_orangepi_experiment.ipynb)
> 返回：[01_robot_system_overview.ipynb](./01_robot_system_overview.ipynb)